In [2]:
# ============================================
# 1. MOUNT GOOGLE DRIVE
# ============================================
from google.colab import drive
drive.mount('/content/drive')

import zipfile
import os

# ============================================
# 2. EXTRACT DATA FROM DRIVE
# ============================================
train_zip_path = '/content/drive/MyDrive/ML Project/GTSRB-Training_fixed.zip'
test_zip_path  = '/content/drive/MyDrive/ML Project/GTSRB_Final_Test_Images.zip'

destination_dir = '/content/sample_data'
os.makedirs(destination_dir, exist_ok=True)

with zipfile.ZipFile(train_zip_path, 'r') as z:
    z.extractall(destination_dir)

with zipfile.ZipFile(test_zip_path, 'r') as z:
    z.extractall(destination_dir)

print("✅ Data Extracted!")

# ============================================
# 3. IMPORT LIBRARIES
# ============================================
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# ============================================
# 4. PATHS
# ============================================
BASE_DIR  = "/content/sample_data/GTSRB"
TRAIN_DIR = os.path.join(BASE_DIR, "Training")

IMG_SIZE    = 100
BATCH_SIZE  = 64
NUM_CLASSES = 43

# ============================================
# 5. DATA GENERATORS
# ============================================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

# ============================================
# 6. MODEL
# ============================================
def build_model():
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    x = layers.Conv2D(16, (3,3), activation='relu')(inputs)
    x = layers.Conv2D(32, (3,3), activation='relu')(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(32, (3,3), activation='relu', padding='same')(x)
    x = layers.Conv2D(64, (5,5), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.BatchNormalization()(x)

    x = layers.Conv2D(64, (3,3), activation='relu', padding='same')(x)
    x = layers.Conv2D(128, (5,5), activation='relu', padding='same')(x)
    x = layers.Conv2D(128, (5,5), activation='relu', padding='same')(x)

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(1024, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.5)(x)

    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    return models.Model(inputs, outputs)

model = build_model()
model.summary()

# ============================================
# 7. COMPILE
# ============================================
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ============================================
# 8. CALLBACKS (SAVE TO DRIVE)
# ============================================
checkpoint_path = '/content/drive/MyDrive/ML Project/best_gtsrb_model.keras'

checkpoint = ModelCheckpoint(
    checkpoint_path,
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3
)

# ============================================
# 9. TRAIN
# ============================================
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=80,
    callbacks=[checkpoint, early_stop, reduce_lr]
)

# ============================================
# 10. SAVE FINAL MODEL
# ============================================
final_model_path = '/content/drive/MyDrive/ML Project/final_gtsrb_model.keras'
model.save(final_model_path)

print("✅ Final model saved!")

# ============================================
# 11. CONVERT TO TFLITE
# ============================================
best_model = tf.keras.models.load_model(checkpoint_path)

converter = tf.lite.TFLiteConverter.from_keras_model(best_model)

# Optimization for speed & size
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

tflite_path = '/content/drive/MyDrive/ML Project/gtsrb_model.tflite'

with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

print("✅ TFLite model saved!")

# ============================================
# 12. PLOT RESULTS
# ============================================
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.legend(['Train', 'Validation'])
plt.title("Accuracy")
plt.show()

# ============================================
# 13. VERIFY TFLITE MODEL
# ============================================
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()

print("✅ TFLite model loaded successfully!")

ModuleNotFoundError: No module named 'google'